<a href="https://colab.research.google.com/github/Gandata/Deep_learning_project/blob/dev%2Fenc-mlp/02_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Feature Extraction

This notebook extracts Concerto Small features for S3DIS Area 5. It is designed to run on a Google Colab T4 instance.

Ensure you have access to the shared Google Drive with the preprocessed data.

### 1. Setup & Mount Drive
Use the project branch that contains the current encoder/extraction code.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

Mounted at /content/drive
Cloning into '/content/Deep_learning_project'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 189 (delta 101), reused 129 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 30.77 MiB | 29.81 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content/Deep_learning_project
Branch 'dev/enc-mlp' set up to track remote branch 'dev/enc-mlp' from 'origin'.
Switched to a new branch 'dev/enc-mlp'
From https://github.com/Gandata/Deep_learning_project
 * branch            dev/enc-mlp -> FETCH_HEAD
Already up to date.
dev/enc-mlp
ae860c3 (HEAD -> dev/enc-mlp, origin/dev/enc-mlp) Changes to try to parallelize preprocessing


### 2. Symlink Data & Checkpoints
We map the Drive folders directly into the repository so our scripts can find them.

In [4]:
# Symlink Drive data
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

### 3. Check the Preprocessed S3DIS Input
If `Area_5` is missing here, extraction cannot start yet.


In [ ]:
!du -sh /content/drive/MyDrive/DL_Project/data/s3dis_processed
!find data/s3dis_processed/Area_5 -maxdepth 1 -type f | head


### 4. Setup Hugging Face Token
Run one of the next two cells. The token is used for the Concerto checkpoint download if anonymous access does not work.


In [5]:
# If token is in colab secrets

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

In [ ]:
# If it is not on colab secrets, paste it
import getpass
import os

print("Enter your Hugging Face token (it will not be saved in the notebook):")
hf_token = getpass.getpass()

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={hf_token}\n")

### 5. Install the Python 3.10 / Concerto Stack
These commands are kept from Ricardo's working Colab setup.


In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10


In [28]:
!python3.10 -m pip install --no-cache-dir --force-reinstall torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124
!python3.10 -m pip uninstall -y torch-scatter || true
!python3.10 -m pip install --no-cache-dir --force-reinstall torch-scatter -f https://data.pyg.org/whl/torch-2.5.0+cu124.html

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.5.0%2Bcu124-cp310-cp310-linux_x86_64.whl (908.3 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0
    Uninstalling torch-2.11.0:
      Successfully uninstalled torch-2.11.0
Looking in links: https://data.pyg.org/whl/torch-2.5.0+cu124.html


In [11]:
!python3.10 -m pip install --no-cache-dir --force-reinstall spconv-cu120
!python3.10 -m pip install --no-cache-dir huggingface_hub timm python-dotenv addict scipy numpy==1.26.4

Looking in links: https://data.pyg.org/whl/torch-2.5.0+cu124.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 96.2 MB/s  0:00:00
  Using cached open3d-0.19.0-cp310-cp310-manylinux_2_31_x86_64.whl.metadata (4.3 kB)
  Using cached fast_pytorch_kmeans-0.2.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached camtools-0.1.8-py3-none-any.whl.metadata (12 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached trimesh-4.12.2-py3-none-any.whl.metadata (13 kB)
  Using cached gradio-6.14.0-py3-none-any.whl.metadata (17 kB)
  Using cached dash-4.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metada

In [14]:
!python3.10 -c "import torch, torch_scatter, dotenv; print('torch', torch.__version__, torch.version.cuda); print('torch_scatter ok:', torch_scatter.__file__); print('dotenv ok')"

  Using cached open3d-0.19.0-cp310-cp310-manylinux_2_31_x86_64.whl.metadata (4.3 kB)
  Using cached fast_pytorch_kmeans-0.2.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached addict-2.4.0-py3-none-any.whl.metadata (1.0 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached camtools-0.1.8-py3-none-any.whl.metadata (12 kB)
  Using cached natsort-8.4.0-py3-none-any.whl.metadata (21 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached trimesh-4.12.2-py3-none-any.whl.metadata (13 kB)
  Using cached gradio-6.14.0-py3-none-any.whl.metadata (17 kB)
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached dash-4.1.0-py3-none

In [8]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto || true


Cloning into '/content/Concerto'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 74 (delta 10), reused 23 (delta 9), pack-reused 42 (from 1)
Receiving objects: 100% (74/74), 2.12 MiB | 28.95 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/Concerto


### 6. Smoke Test the Runtime
Check `spconv`, `concerto`, and the project encoder before the full extraction.


In [ ]:
!python3.10 -c "import sys, torch; sys.path.insert(0, '/content/Concerto'); import concerto; import spconv.pytorch as spconv; print('python ok'); print(torch.__version__, torch.version.cuda); print(spconv.__file__)"
!PYTHONPATH=/content/Deep_learning_project:/content/Concerto CONCERTO_DIR=/content/Concerto python3.10 -c "import torch; from src.encoder import ConcertoEncoder; device = 'cuda' if torch.cuda.is_available() else 'cpu'; encoder = ConcertoEncoder(device=device); sample = {'coord': torch.randn(256, 3), 'color': torch.rand(256, 3)}; features = encoder(sample); print('encoder forward ok:', tuple(features.shape), features.dtype, features.device, torch.isfinite(features).all().item())"


### 7. Run Feature Extraction
This calls the real extraction script and saves compact training-ready `.npz` files to `features/s3dis_area5`.
Features are stored as `float16`, labels as `uint8`, and coordinates are omitted to keep files much smaller.
If you already have older large `.npz` files, rerun with `--overwrite` or delete them first so they get regenerated in the compact format.


In [ ]:
!PYTHONPATH=/content/Deep_learning_project:/content/Concerto CONCERTO_DIR=/content/Concerto python3.10 /content/Deep_learning_project/scripts/extract_features.py --data_dir /content/Deep_learning_project/data/s3dis_processed --out_dir /content/Deep_learning_project/features/s3dis_area5 --feature-dtype float16
